# Clasificación con MiniLM

Ignorar, fue una prueba de clasificación temprana

In [ ]:
import pandas as pd
import re
import unicodedata
import torch
from sentence_transformers import SentenceTransformer, util

# ==============================================================================
# FASE 0: PREPROCESAMIENTO, LIMPIEZA Y NORMALIZACIÓN
# ==============================================================================
print(">>> FASE 0: Limpiando el dataset de ruidos y nulos...")
ARCHIVO_ENTRADA = "dataset_buap_FINAL_recuperado.csv"
ARCHIVO_SALIDA = "corpus_clasificado_ES_GPU.csv"

def normalizar_texto(texto):
    """Elimina acentos y convierte a minúsculas para mejorar la detección con Regex."""
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

try:
    df = pd.read_csv(ARCHIVO_ENTRADA)
    # 1. Eliminar valores nulos
    df = df.dropna(subset=['Texto_Comentario'])
    
    # 2. Limpieza de menciones redactadas
    df = df[df['Texto_Comentario'].str.strip() != "<redacted_mention>"]
    
    # 3. Eliminar comentarios que son solo emojis o símbolos
    df = df[df['Texto_Comentario'].str.contains(r'[a-zA-Z0-9]', regex=True)]
    
    # 4. Eliminar duplicados
    df = df.drop_duplicates(subset=['Texto_Comentario'])
    print(f"[*] Limpieza completada. Registros útiles para procesar: {len(df)}")
except Exception as e:
    print(f"[X] Error en carga/limpieza: {e}")

# ==============================================================================
# FASE 1: CONFIGURACIÓN DE GPU Y CARGA DE MODELO
# ==============================================================================
print(">>> FASE 1: Configurando GPU Tesla T4 y cargando modelo...")
device = "cuda" if torch.cuda.is_available() else "cpu"
RUTA_MODELO = '/home/jovyan/huggingface/sentence-transformers/all-MiniLM-L6-v2/main'
modelo_sbert = SentenceTransformer(RUTA_MODELO, device=device)

# ==============================================================================
# FASE 2: DICCIONARIO SEMÁNTICO
# ==============================================================================
taxonomia_semantica = {
    # 1. TAXONOMÍA: SEXOTIPOS
    "TAX_SEX_Sexualidad": "loca puta zorra golfa ramera cuzca piruja urgida fácil cabrona cualquiera sidosa",
    "TAX_SEX_Anatomia_Genital": "verga huevos pelotas pussy culo reata pito concha pichula pico raja escroto",
    "TAX_SEX_Filiacion": "chinga tu madre bastardo malparido aborto hijo de perra hijo de puta conchetumare joputa",
    "TAX_SEX_Orientacion": "joto maricón lesbiana machorra lencha puñal afeminado mayate puto traga sables fleto cacorro",

    # 2. TAXONOMÍA: ONTOTIPOS
    "TAX_ONT_Capacidad_Intelectual": "menso estúpido tonto tarado analfabeto pendejo imbécil retrasado mongol idiota sordo ciego autista baboso",
    "TAX_ONT_Caracter": "inútil vago flojo huevón chillón conchudo mantenido zángano parasito fracasado mediocre",
    "TAX_ONT_Conducta": "vicioso holgazán hocicón patán culero inmaduro mandilón parásito mierda basura escoria",
    "TAX_ONT_Moralidad": "mala persona hipócrita sinvergüenza víbora doble cara tóxica lamebotas traidor ratero ladrón chupasangre",
    "TAX_ONT_Apariencia_Fisica": "gordo feo cuatro ojos chaparro buchona enano horrible monstruo esqueleto ballena cerdo mutante",
    "TAX_ONT_Forma_Vestir": "pandroso fodonga mugrosa fachuda piojoso harapiento cholo naco chusma brayan kevin",
    "TAX_ONT_Edadismo": "vieja anciano ruca rabo verde mocoso chavorruco fósil sugar momia decrepito vejestorio",
    "TAX_ONT_Afectividad_Empatia": "egoísta corazón de piedra témpano ojete sin madre frívola desalmado insensible sociópata",
    "TAX_ONT_Zoomorfismo_Animalizacion": "cerda vaca simio zorra burro bestia gata perro chango rata cerdo mono gusano víbora",
    "TAX_ONT_Escatologia": "cagar mierda pedo escoria caca mojón surrar vomito diarrea",

    # 3. TAXONOMÍA: SOCIOTIPOS Y ETNOTIPOS
    "TAX_SOC_Estatus_Social": "naco pobretón jodido fresa chusma godín muerto de hambre mirrey pordiosero mendigo",
    "TAX_SOC_Ideologia": "chairo comunista progre fifí porro woke feminazi facho derechairo conservador libtard",
    "TAX_SOC_Profesion": "transa tira nini lacra chacha puercos prostituta ramera matasanos leguleyo",
    "TAX_ETN_Racismo": "color cartón prieto indio bajado del cerro negro ario cobrizo chango macaco sudaca",
    "TAX_ETN_Xenofobia": "gachupín pipope oaxaco chilango gringo veneco bolita saltamuros yankee",

    # 4. RAPPORT MANAGEMENT (Cortesía y Sociabilidad)
    "RAP_IMG_Calidad": "ataque a atributos personales inteligencia inútil vaca gorda incompetente bueno para nada estúpido ignorante",
    "RAP_IMG_Identidad_Social": "ataque identidad social grupo indio naco fresa godinez plebe chusma escoria social",
    "RAP_IMG_Relacion": "incumple expectativas lealtad tonto barco lamebotas traidor judas falso mentiroso vendido",
    "RAP_SOC_Igualdad_Equidad": "impone poder niega trato justo me la pelas cállate obedeces arrodíllate calla esclavo",
    "RAP_SOC_Asociacion": "rompe afiliación excluye lárgate no te queremos progre proletario fuera vete no perteneces",

    # 5. ESTRATEGIAS INDIRECTAS Y HUMOR
    "MEC_IND_Humor": "mira el genio darwin te habría usado burla comedia chiste payaso bufón hazmerreír",
    "MEC_IND_Ironia": "eres un genio modo ahorro energía mente galáctica gracias por tu sesuda aportación brillante lumbrera einstein",
}

# ==============================================================================
# FASE 3: REGLAS ESTRUCTURALES Y SINTÁCTICAS (SOLO ESPAÑOL)
# ==============================================================================
print(">>> FASE 3: Ejecutando detección estructural exhaustiva...")

def clasificar_estructuras(texto_original):
    t = normalizar_texto(texto_original)

    # --- NÚCLEOS DEL INSULTO DIRECTO ---
    # Captura variaciones
    mec_dir_acto_explicito = 1 if re.search(r"\b(analfabet[oa]s?|nac[oa]s?|estupid[oa]s?|p[u\*]t[oa]s?|mierd[a@]|idiotas?|pendej[oa]s?|imbeciles?)\b", t) else 0
    mec_dir_declarativo = 1 if re.search(r"\b(eres|es|sos)\s+(un|una|bien|muy)\b", t) else 0
    mec_dir_rechazo = 1 if re.search(r"\b(jodete|muerete|pudrete|no mames|no chingues)\b", t) else 0
    mec_dir_metafora = 1 if re.search(r"\b(vete a ladrar|vete a llorar|vete a pastar|vete a la verga|vete a la mierda)\b", t) else 0
    mec_dir_silencio = 1 if re.search(r"\b(callate|cierra el hocico|no chilles|cerrando el ortito)\b", t) else 0
    mec_dir_alejamiento = 1 if re.search(r"\b(vete al diablo|largate|pirate|esfumate|muevete|a llorar a otro lado)\b", t) else 0
    mec_dir_abreviaturas = 1 if re.search(r"\b(ptm|alv|vrga|ctm|hdtpm|valv|mms|lptm)\b", t) else 0

    # --- NÚCLEOS DEL INSULTO INDIRECTO ---
    mec_ind_modalidad = 1 if re.search(r"¿.*?(cerebro|piensas|tonto|inteligente|pendej[oa]).*?\?", t) else 0
    mec_ind_lexico = 1 if re.search(r"\b(pareces|como un|parece)\s+(perro|mula|parasito|tortuga)\b", t) else 0
    mec_ind_deixis = 1 if re.search(r"\b(la jodiste|me la pelas|chinga la tuya|la cagaste)\b", t) else 0
    mec_ind_eufemismo = 1 if re.search(r"\b(tinaco|chingaquedito|tiznada|guayaba|kiwis|china)\b", t) else 0
    mec_ind_omision = 1 if re.search(r"\b(ya valio|vete a la\.\.\.|no ma\.\.\.|hijo de su\.\.\.)\b", t) else 0
    mec_ind_posesivo = 1 if re.search(r"\b(tu abuela|tu culo|tu pedo|tu madre|tu pito)\b", t) else 0

    # --- MODIFICADORES E INTENSIFICADORES (NO SON INSULTOS POR SÍ SOLOS) ---
    int_cuantificadores = 1 if re.search(r"\b(bien|rete|super|tan)\b", t) else 0
    int_colectivos = 1 if re.search(r"\b(puros|bola de|un buen de)\b", t) else 0
    int_exclamativos = 1 if re.search(r"¡(que|como)\b", t) else 0
    int_comp_hiperbolicas = 1 if re.search(r"\b(mas.*que)\b", t) else 0
    int_comparaciones = 1 if re.search(r"\b(como)\b", t) else 0
    int_subjuntivo = 1 if re.search(r"\b(ojala|que te)\b", t) else 0
    int_imperativo = 1 if re.search(r"\b(vete a)\b", t) else 0

    # --- ESTRATEGIAS (ALGUNAS NO SON INSULTOS POR SÍ SOLAS) ---
    est_vocativo = 1 if re.search(r"^(put[oa]|asqueros[oa]|estupid[oa]|imbecil|analfabeto|nac[oa]|ridicul[oa])\b", t) else 0
    est_asercion = 1 if re.search(r"\b(eres verdaderamente|no puedes hacer nada bien)\b", t) else 0
    est_ref_personal = 1 if re.search(r"\b(tu boca|me das asco|no te sabes vestir)\b", t) else 0
    est_ref_tercera = 1 if re.search(r"\b(esa tonta|ella esta|lo dice el anciano)\b", t) else 0
    est_reforzadores = 1 if re.search(r"\b(me entiendes|te queda claro|para que te quede claro)\b", t) else 0
    est_silenciadores = 1 if re.search(r"\b(callate|cierra el pico|cerrando el pico)\b", t) else 0
    est_despidos = 1 if re.search(r"\b(te me vas|abrete|largate|desaparecete|llegale)\b", t) else 0

    return pd.Series([
        mec_dir_acto_explicito, mec_dir_declarativo, mec_dir_rechazo, mec_dir_metafora, 
        mec_dir_silencio, mec_dir_alejamiento, mec_dir_abreviaturas,
        mec_ind_modalidad, mec_ind_lexico, mec_ind_deixis, mec_ind_eufemismo, mec_ind_omision, mec_ind_posesivo,
        int_cuantificadores, int_colectivos, int_exclamativos, int_comp_hiperbolicas, 
        int_comparaciones, int_subjuntivo, int_imperativo,
        est_vocativo, est_asercion, est_ref_personal, est_ref_tercera, 
        est_reforzadores, est_silenciadores, est_despidos
    ])

# Nombres exactos de las columnas estructurales
columnas_nucleo_estructural = [
    'MEC_DIR_Acto_Explicito', 'MEC_DIR_Enunciado_Declarativo', 'MEC_DIR_Acto_Rechazo', 'MEC_DIR_Metafora',
    'MEC_DIR_Mandato_Silencio', 'MEC_DIR_Alejamiento', 'MEC_DIR_Abreviaturas',
    'MEC_IND_Cambio_Modalidad', 'MEC_IND_Contenido_Lexico', 'MEC_IND_Deixis', 'MEC_IND_Eufemismo', 'MEC_IND_Omision', 'MEC_IND_Posesivo',
    'EST_Vocativo_Negativo', 'EST_Asercion_Negativa', 'EST_Ref_Personalizada', 'EST_Ref_Tercera', 
    'EST_Silenciadores', 'EST_Despidos'
]

columnas_modificadores = [
    'INT_Cuantificadores', 'INT_Colectivos', 'INT_Exclamativos', 'INT_Comp_Hiperbolicas', 
    'INT_Comparaciones', 'INT_Subjuntivo', 'INT_Imperativo', 'EST_Reforzadores'
]

todas_las_estructuras = columnas_nucleo_estructural + columnas_modificadores

# Procesamiento rápido de reglas lógicas
df[todas_las_estructuras] = df['Texto_Comentario'].apply(clasificar_estructuras)

# ==============================================================================
# FASE 4: CLASIFICACIÓN SEMÁNTICA ACELERADA POR GPU (SBERT)
# ==============================================================================
print(">>> FASE 4: Procesando semántica y taxonomía (Batch en GPU)...")

nombres_semanticos = list(taxonomia_semantica.keys())
vectores_taxo = modelo_sbert.encode(list(taxonomia_semantica.values()), convert_to_tensor=True)

embeddings_comentarios = modelo_sbert.encode(
    df['Texto_Comentario'].tolist(), 
    batch_size=64, 
    convert_to_tensor=True,
    show_progress_bar=True
)

similitudes = util.cos_sim(embeddings_comentarios, vectores_taxo)

# Establecemos un umbral de similitud para activar las etiquetas semánticas
# Entre más alto el umbral, más estricta la detección (menos falsos positivos pero más falsos negativos)
UMBRAL = 0.60 
for i, nombre in enumerate(nombres_semanticos):
    df[nombre] = (similitudes[:, i] >= UMBRAL).cpu().numpy().astype(int)

# ==============================================================================
# FASE 5: ETIQUETA DE CONTROL Y LIMPIEZA DE FALSOS POSITIVOS
# ==============================================================================
# Las columnas núcleo son las que definen si un comentario es insulto o no
columnas_nucleo_total = columnas_nucleo_estructural + nombres_semanticos

# Es "Insulto" si tiene alguna etiqueta NÚCLEO activada
df['Es_Insulto'] = (df[columnas_nucleo_total].sum(axis=1) > 0).astype(int)

# Eliminamos los Falsos Positivos de los modificadores (ej. detectar "bien" en un comentario amable)
for col in columnas_modificadores:
    df.loc[df['Es_Insulto'] == 0, col] = 0

df.to_csv(ARCHIVO_SALIDA, index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print(f"[*] PROCESO FINALIZADO CON ÉXITO EN GPU.")
print(f"[*] Comentarios neutros detectados (No Insulto): {(df['Es_Insulto'] == 0).sum()}")
print(f"[*] Comentarios con insulto detectados: {df['Es_Insulto'].sum()}")
print(f"[*] Resultados guardados en: {ARCHIVO_SALIDA}")
print("="*60)

# Clasificación con diccionario de lexemas

In [ ]:
import pandas as pd
import re
import unicodedata

# ==============================================================================
# FASE 0: PREPROCESAMIENTO, LIMPIEZA Y NORMALIZACIÓN
# ==============================================================================
print(">>> FASE 0: Limpiando el dataset de ruidos y nulos...")
ARCHIVO_ENTRADA = "dataset_buap.csv"
ARCHIVO_SALIDA = "corpus_clasificado_buap.csv"

diccionario_emojis_prioridad = {
    # Risas y Burla
    "😂": " jajaja ",
    "🤣": " jajaja ",
    "💀": " me muero ",
    "😭": " lloron ",
    "🤡": " payaso ",
    "🥱": " aburrido ",

    # Escatología y Asco
    "💩": " mierda ",
    "🤮": " vomito ",
    "🤢": " asqueroso ",
    "🗑️": " basura ",
    "🗑": " basura ",

    # Animalización / Insultos
    "🐒": " chango ",
    "🦍": " gorila ",
    "🐷": " cerdo ",
    "🐖": " puerco ",
    "🐀": " rata ",
    "🐄": " vaca ",
    "🐶": " perro ",
    "🐍": " vibora ",
    "🐑": " borrego ",

    # Agresión y Gestos
    "🖕": " vete a la mierda ",
    "🤬": " carajo ",
    "😡": " coraje ",
    "🤫": " callate ",

    # Intelecto
    "🧠": " cerebro ",
    "🤓": " nerd ",
}

def contiene_texto_o_emoji_prioritario(texto):
    t = str(texto)
    tiene_alfanumerico = bool(re.search(r'[a-zA-Z0-9]', t))
    tiene_emoji_prioritario = any(emoji in t for emoji in diccionario_emojis_prioridad)
    return tiene_alfanumerico or tiene_emoji_prioritario

def normalizar_texto(texto):
    """
    Elimina acentos y convierte a minúsculas. 
    Mantiene símbolos como @ o * para detectar evasiones de censura.
    """
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def sustituir_emojis_prioridad(texto):
    t = str(texto)
    for emoji, reemplazo in diccionario_emojis_prioridad.items():
        t = t.replace(emoji, reemplazo)
    return t

try:
    df = pd.read_csv(ARCHIVO_ENTRADA)
    df = df.dropna(subset=['Texto_Comentario'])
    df = df[df['Texto_Comentario'].str.strip() != "<redacted_mention>"]
    df = df[df['Texto_Comentario'].apply(contiene_texto_o_emoji_prioritario)]
    df = df.drop_duplicates(subset=['Texto_Comentario'])
    print(f"[*] Limpieza completada. Registros útiles para procesar: {len(df)}")
except Exception as e:
    print(f"[X] Error en carga/limpieza: {e}")
    df = pd.DataFrame({"Texto_Comentario": ["Eres bien pendejo", "Qué bonita facultad", "Pinche p*to"]})

# ==============================================================================
# FASE 1: DICCIONARIO DE LEXEMAS
# ==============================================================================
print(">>> FASE 1: Construyendo reglas basadas en lexemas para 67 categorías...")

diccionario_lexemas = {
    # --- MECANISMOS DE CONSTRUCCIÓN ---
    "Acto_explicito": r"\b(analfabet[oa]s?|nac[oa]s?|estupid[oa]s?|p[\*u]t[oa]s?|mierd[a@s]|lelos?|retrasad[oa]s?|incult[oa]s?|idiotas?|mongol(es)?|pendej[oa]s?|imbecil(es)?|babos[oa]s?|pinches?|tarad[oa]s?|cabron(es)?|culer[oa]s?|o[gj]etes?)\b",
    "Enunciados_declarativos_apelativos": r"\b(eres un|eres una|sos un|sos una|vales)\s+(pendej[oa]s?|mediocres?|huevon(es)?|inutil(es)?|mierda|basura|porqueria|estupid[oa]|idiota|babos[oa])|(eres pendej[oa]|humillate|vales madres?|no sirves( para nada)?)\b",
    
    # Nuevas subcategorías de Rechazo
    "Actos_directivos_de_rechazo_con_formulas_imprecatorias": r"\b(jodet[ea]|mueret[ea]|pudret[ea]|ojala (te maten|te mueras)|que te (viole un burro|parta un rayo|cargue el payaso))\b",
    "Actos_directivos_de_rechazo_con_negaciones_metaforicas": r"\b(no (mames|chingues|jodas)|no (te pases|mames) de verga|no estes (chingando|de castroso))\b",
    "Actos_directivos_de_rechazo_con_metaforas_directas": r"\b(vete a( la)? (verga|mierda|fregada|chingada|puta|cola|versh)|vete a (ladrar|llorar(le)?|pastar|chingar( a tu madre)?)|vete al (diablo|carajo|infierno))\b",
    "Actos_directivos_de_rechazo_con_mandato_de_silencio": r"\b(callate (el hocico|el pico|la boca|estupid[oa])|cierra (el hocico|el pico|la boca)|deja de (ladrar|llorar)|no chilles|no hables tan fuerte|vamos cerrando el ortito|no digas mamadas|tanto pedo para cagar aguado)\b",
    "Actos_directivos_de_rechazo_con_exhortaciones_de_alejamiento": r"\b(largate|pirate|esfumate|muevete baboso|ya (te estas yendo|vete)|no te quiero ver|pelale( a la verga)?|a la chingada|vete a cagar el palo|a llorar a otro lado|llegale|abrete)\b",
    
    "Abreviaturas": r"\b(ptm|alv|nye|vrga|sptm|vv|brga|vlg|nmms|nmm|ctm|chtm|hdspm|hdtpm|valv|mms|lptm|pto|pta|hspm|hdpt?|chsm|dlv|cchm|hdtsptm|kks?|mmda?|kbrn|qlo|cul|ogt)\b",
    "Cambio_de_modalidad": r"¿.*?(cerebro|piensas|tonto|pendej[o@a]s?|intelectual|carbura|san pendejo|haces|sordo|ciego|idiota|estupid[oa]).*?\?",
    "Contenido_lexico": r"\b(pareces (un perro|una vaca|cerdo|mula)|pinche perrit[oa]|mas terc[oa] que una mula|eres un parasito|le gana una tortuga|cierra el (hocico|pico)|levanta las patas|abre las patas|lamebotas|((solo )?andas )?mamando|cara de (perro|artesania|culo|pendejo|verga))\b",
    "Deixis_discursiva": r"\b(ya (la )?(jodiste|cagaste|cag[oó])|me la pelas|chinga la tuya|la hiciste de p[e3]do|la tuya|te brilla el cozo|tu culo)\b",
    "Eufemismos": r"\b(no le (sube el agua|gira la ardilla)|chingaquedito|esta bien menso|hijo de la (tiznada|guayaba|china|madrugada|fregada|chingada)|patada en los (kiwis|huevos)|estas bien (cutre|curios[oa])|pendecto|un asco|quema mucho el sol|vete a la (china|versh|verr|fregada)|verijas?)\b",
    "Omision_de_la_groseria": r"\b(ya vali[oó]|chinga \.\.\.|vete a la( \.\.\.|\.\.\.)|no ma(\.\.\.|ches|chen|nches)|no friegues|ya bail[oó]|pen\.\.\.|hijo de su\.\.\.|a la \.\.\.|chin\.\.\.|si, caray|hijo de la reverenda|me lleva la)\b",
    "Posesivos_enfatizadores": r"\b(tu (abuela|mama|madre|jefa|culo( en bicicleta)?|p[e3]do|cola|pito|pendej[oa]|cara|hocico)|t[u] put[a@] madr[e3]|su (reputisima|puta) madre|tus (nalgas|huevos|mamadas|pendejadas))\b",
    "Humor": r"\b(genio de la familia|dej[ea]s? de ser tan estupid[oa]s?|darwin te habria usado|inteligencia te persigue|no te creas (un culo|la gran cosa|muy vrg)|para que tienen culos|hijo de hiena|dios te guarde( y se le olvide donde)?|envidia de tu boca|pintar no te gusta|cazabamos mamuts|te estaba clonando|escupes pendejadas|premio nobel de pendejez)\b",
    "Ironia_polar_o_contrastiva": r"\b(eres un genio|le sabes|ahorro de energia|nieve en gaza|coeficiente intelectual|neuronas( se han)? ido de vacaciones|nube cuando te vas|universo( esta)? conspirando|ese perro de quien es)\b",
    "Ironia_de_personificacion": r"\b(mi metabolismo|el mundo esta contra (ti|mi)|europa cada 80 a[nñ]os|universo conspirando|mi algoritmo dijo|diosito te hizo con flojera)\b",
    "Ironia_de_falsa_cortesia": r"\b(ha vuelto el rey|la de pensar no( te la)? sabes|sesuda aportacion|mente galactica|trabajo de dios|bloquear a las cuentas|rusarin|todo un genio|un tonto a las tres|iq (mayor|promedio)|que iluminad[oa])\b",
    
    # --- TAXONOMÍA (SEXOTIPOS Y ONTOTIPOS) ---
    "Sexualidad_Mujer": r"\b(locas?|putas?|zorras?|golfas?|urgidas?|cuzcas?|zopilotas?|guarras?|rameras?|catadora(s)? de (pitos|vrgs)|andas therian|peponas?|cola suelta|golosas?|culo abierto|andas ovulando|maracas?|gata rompe hogares|prostipirugolfas?|pirujas?|putigolfas?|tijeras?|lenchas?|machorras?|lesbianas?|marimachas?|bolleras?|busconas?|ofrecidas?|facilonas?|pagar con cama)\b",
    "Sexualidad_Hombre": r"\b(putos?|jotos?|afeminados?|amanerados?|maripos[oa]s?|homosexual(es)?|homosexgey|jotas?|sumis[oa]s?|pasiv[oa]s?|maricon(es)?|maricas?|princesos?|puñal(es)?|pocos huevos|chotos?|churpios?|cochinos?|depravados?|culeros?|mayates?|muerdealmohadas|soplaga[yi]tas|tragables?)\b",
    "Sexualidad_Persona_no_binaria": r"\b(niño-niña|jochis|calados?|hetere|milf|niñis|chica marciana|trapos?|desviad[oa]s?|fenomenos?|engendros?|mutantes?|Transformers?)\b",
    "Anatomia_genital": r"\b(me vale (verga|madre)|chupa(me)? la (verga|pija)|te faltan huevos|no tienes pelotas|ya cogetelo|callate el hocico|chupamela|por tus ovarios|mandalo a la verga|meme vergue|vele vergue|te dolio la pussy|te ardio culer[oa]|zumbale a la reata|caele a la verga|picate el hoyo|tu cola|chucha|puchaina|trans con pistola|me pelas( toda la)? verga|pitos?|culos?|huevos?|vergas|conchas?|pichulas?|pijas?|nepe|panochas?|chichis?|tetas?|chocho)\b",
    "Filiacion": r"\b(bastardos?|malparidos?|aborto(s)?( de la naturaleza| fallido| malogrado)?|malnacidos?|no deseados?|hijos? de (la chingada|puta|perra|su puta madre|mil putas|la gran puta))\b",
    "Capacidad_intelectual": r"\b(mens[oa]s?|estupid[oa]s?|tont[oa]s?|tarados?|necios?|cabeza hueca|analfabetos?|babos[oa]s?|weys?|g[u]eys?|falta de acido folico|incompetentes?|animal(es)?|fracasados?|pendej[oa@e]s?|estepedes?|totos?|imbecil(es)?|mongolit[oa]s?|retrasad[oa]s?|totol(es)?|incapaz|soquetes?|san pendejo|san idiota|re wey|retrasadit[oa]s?|zopencos?|zopes?|ignorantes?|bestias?|asnos?|mentecatos?|descerebrados?|papanatas?|burros?|inutiles?|cegatos?|miope de mierda)\b",
    "Caracter": r"\b([jg]etas?|inutil(es)?|vagos?|flojos?|conchudos?|chillon(es)?|huevon(es)?|weon(es)?|intens[oa]s?|migajeros?|mecha corta|mantenidos?|zangano(s)?|arrastrados?|arrastrarse|chupopteros?|gorrones?|aprovechados?|dejados?|agachon(es)?|cobardes?|zacatones?|rajones?|llorones?|narcisistas?|mediocres?|apaticos?)\b",
    "Conducta": r"\b(locos?|[jg]eton|viciosos?|holgazan(es)?|fodong[oa]s?|hocicon(es)?|patan(es)?|zopencos?|culeros?|inmaduros?|pelados?|mantenidos?|mandilon(es)?|haragan(es)?|baqueton(es)?|parasitos?|lambiscon(es)?|arrimados?|lamehuevos|cringe|pinches?|malditos?|cabrones?|pasad[oa]s? de lanza|castrosos?|enfadosos?|pesados?|mierderos?|cagason(es)?|mamones?|payasos?|alzad[oa]s?|prepotentes?|soberbios?|arrogantes?|chismosos?|metiches?|mitoteros?|oportunistas?|trepadores?)\b",
    "Moralidad": r"\b(mala persona|lamebotas|viboras?|carroñeros?|hipocritas?|sinverguenzas?|mochos?|sin respeto|judas|doble cara|putridos?|chupasangres?|basura|culer[oa]s?|desgraciad[oa]s?|pinche vieja|asqueros[o@a]s?|toxic[oa]s?|chacales?|tibios?|ghosteadores?|mujeriegos?|mañosos?|serpentes?|brujas?|mustios?|falsos?|mentirosos?|vendidos?|escorias?|rastreros?|corruptos?|tranzas?|desleales?|caraduras?|malagradecidos?|ratas?|ruines?|viles?)\b",
    "Apariencia_fisica": r"\b(gord[oa]s?|fe[oa]s?|cuatro ojos|ñoños?|chaparros?|buchonas?|duendes?|gnomos?|pedazo de seca|cara de olmeca|horribles?|chichi-chistosa|warzone|nahuales?|aimep3|morzas?|ciegos?|pitufos?|piratas?|enanos?|ranci[oa]s?|marihuanos?|cricos[oa]s?|folkloricos?|nopal en la cara|bigote de cantinflas|ogros?|desnalgados?|tablas?|mancos?|muñones?|drogos?|piedreros?|el manotas|olmecas?|chichotas?|minions?|nalgas caidas|sin chichis|adefesios?|mamarrachos?|esperpentos?|bizcos?|tuertos?|cojos?|calvos?|pelones?|marran[oa]s?|puerc[oa]s?|chanch[oa]s?|desnutridos?|narizones?|orejones?|trompudos?|obes[oa]s?|gordolob[oa]s?|tinacos?|rotoplas|escu[aá]lid[oa]s?|esqueletos?)\b",
    "Forma_de_vestir": r"\b(pandrosos?|fodong[oa]s?|mugros[oa]s?|zarrapastrosos?|payasos?|fachud[oa]s?|horripilantes?|dornetos?|cagados?|piojosos?|ahorrador de agua|churpias?|vagos?|teporochos?|borrachos?|chilapastrosos?|harapientos?|nacos?|chusma|guaches?|huarachudos?)\b",
    "Edadismo": r"\b(viej[oa]s?|brujas?|ancian[oa]s?|ruc[oa]s?|rabo verde|mocosos?|viej[oa]s? decrepitos?|momias?|señoras?|viej[oa]s? huangos?|asaltacunas|escuincles?|chilon(es)?|asaltatumbas|sugar(s)?|fosil(es)?|viejo asqueroso|generacion z|mazapan?|reliquias? vivientes?|colageno|chavorruc[@oa]s?|miatos?|chamacos?|nalgas miadas|ruquillos?|caducos?|carcamales?|cagones?|pubertos?|niñatos?|vejetes?|rucas?|dinosaurios?)\b",
    "Afectividad_empatia": r"\b(egoistas?|corazon de piedra|tempanos?|no sientes?|poquita madre|ojetes?|tantita madre|poca madre|puta roca|frivol[oa]s?|vale madres|culeros?|desalmados?|insensibles?|sociopatas?|secos?|amargados?|rencorosos?|envidiosos?|malaleche|crueles?|despiadados?|cogelones?|apaticos?)\b",
    "Zoomorfismo_animalizacion": r"\b(cerdas?|vacas?|ballenas?|simios?|zorr[oa]s?|arpias?|cabron(es)?|zanganos?|burros?|bestias?|cucarach[oa]s?|gat[oa]s?|perr[oa]s?|viboras?|chang[oa]s?|therian|patitos?|tiranosaurios?|puercos?|sanguijuelas?|asnos?|monos?|mexichangos?|venados?|chapulines?|cerdos?|marranos?|rata(s)?( de dos patas)?|lombriz de aguapuerca|parasitos?|gusano(s)?( infeliz)?|macacos?|gorilas?|chimpances?|buitres?|gallinas?)\b",
    "Escatologia": r"\b(cagar|mierd[a@s]|p[e3]dos?|desechos?|escoria|caca|popo|cagados?|mojon(es)?|surrar|nalgas meadas|caca seca|cagadera|vomito|diarrea|cagadas?|meados?|miados?|vomitivos?|pudricion|podridos?|mugre|porqueria|basura|apestosos?|hediondos?|apestados?|kks?|zurullos?|mierderos?)\b",
    "Estatus_social": r"\b(nac[oa]s?|pobretones?|churpias?|rusticos?|jodidos?|mirreyes?|fresas?|chusma|ordinarios?|muertos? de hambre|godin(es)?|clasemedieros?|proletarios?|hambreados?|niñit[oa]s? de las lomas|estudihambres?|olvidados?|inexistentes?|wannabes?|pueblerinos?|del cerro|proles?|buchonas?|bajado(s)? del cerro|pichurrientos?|macuarros?|chundos?|pelados?|gatos?|plebeyos?|garrulos?|corrientes?|vulgares?|pirruris?|muerto de hambre|pordioseros?|mendigos?)\b",
    "Ideologia": r"\b(chairos?|comunistas?|progres?|fifi(s)?|porros?|morenistas?|panistas?|priistas?|amlovers?|derechistas?|simps?|woke|feminazis?|aliades?|whitexicans?|hippies?|fachos?|derechairos?|conservador(es)?|libtards?|pejezombies?|fachistas?|fascistas?|nazis?|rojillos?|zurdos?|cristofascistas?|mochos?|persignados?|machirulos?|onvres?|manginas?|apologistas?|borregos?|paleros?)\b",
    "Profesion": r"\b(transas?|tiras?|ninis?|lacras?|chachas?|puercos?|mañosos?|maña|pobresor|prostitutas?|matasanos?|leguleyos?|albañiles?|chalanes?|criadas?|sirvientas?|gatas?|taxistas?|franeleros?|tamaleros?|taqueros?|viene vienes?|polizontes?|chotas?|cuicos?|huelepegas|teporochos?|guaruras?)\b",
    "Racismo": r"\b(color carton|priet[oa]s?|nopales?|indios?|mejorar la raza|bajado del cerro( a tamborazos)?|color humilde|pueblerinos?|rural(es)?|aborigen(es)?|apaches?|color mole|arios?|chocorrol(es)?|duvalines?|dubalines?|tono de bajos recursos|tono de piel|color lenteja|negros?|color canasta basica|mongolit[oa]s?|llanta quemada|g[u]erit[oa]s?|silvestres?|autoctonos?|semibronceados?|nahuales?|tez humilde|frijoleros?|chapulineros?|nopaleros?|huarachudos?|color llanta|oaxacos?|macacos?|chimpances?|simios?|indigenas?|tarahumaras?)\b",
    "Xenofobia": r"\b(delincuentes?|tlaxcala no existe|gachupines?|pipopes?|oaxacos?|chilangos?|gringos?|franchutes?|gallegos?|poblanitos?|sureños?|coreanitos?|chiapanecos?|ilegales?|saltamuros?|jarochos?|peruanos?|bolivianas?|chinitos?|tlaxcalita|sudacas?|venecos?|yankees?|mojados?|espaldas mojadas?|hambrientinos?|panchitos?|judios?|gabachos?|bolillos?|yolandas?|guatemaltecos?|macacos?|macuis?)\b",
    
    # --- MODIFICADORES E INTENSIFICADORES ---
    "Cuantificadores_Adverbios_Cualidad": r"\b(?:bien|buen[oa]|re(?:te|quet[ea]?|t[oa]?)?|requet[ea]?|s[u]per|hiper|h[i]per|mega|ultra|archi|muy|demasiad(?:o|a|os|as)|tan|tremend[oa]mente|sumamente|excesiv(?:o|a)s?|bastant(?:e|es)|sobr(?:e|ad[oa]))\b",
    "Cuantificadores_Sustantivos_colectivos_o_determinantes_numerales_Entidad": r"\b(?:puros?\s+pinch(?:es|i)?s?|sarta\s+de|bola\s+de|un\s+(?:buen|gran)\s+(?:mont[o]n|chingo(?:n|es)?)(?:\s+de)?|mont[o]n(?:\s+de)?|chorro\s+de|horda\s+de|manada\s+de|pandilla(?:\s+de)?|legi[o]n(?:\s+de)?|grupo\s+de|unos?\s+(?:cuantos|pocos|varios)|varios?\s+de|much(?:o|os|as)|un\s+puñado\s+de)\b",
    "Determinantes_exclamativos": r"(?:[¡!¿?]*\b(?:que|como|cuanto|cuanta|cuantos|cuantas|cual|cuales|vaya)\b)",
    "Comparaciones_hiperbolicas": r"\b(?:m[a]s(?:\s+[\waeiouñ]{1,25}){1,6}\s+que|como\s+su\s+(?:pap[a]|mam[a]|madre)|como\s+un[ao]?\s+(?:piedra|perro|pendej[oa]|idiota|mono|cerdo|burro|basura|muerto)|igual(?:ito|itos)?\s+a|tal\s+como|pareces?\s+(?:un[ao]?\s+|como\s+un[ao]?))\b",
    "Subjuntivo": r"\b(?:ojal[a](?:\s+y|\s+que)?|que\s+te\s+(?:partan|maten|muera[sn]?|mueras|pudras|fallezcas|jodan|rompan|corten|lleve[n]?|cargue[n]?|arruinen|te\s+deje[n]?|\bte\s+viole[n]?)|ojal[a]\s+te)\b",
    "Imperativo": r"\b(?:jodet[ea]|mueret[ea]|pudret[ea]|vete|chinga|callate|cierra|larga|pira|esfuma|mueve|abre|sal|no (mames|chingues|jodas|chilles|hables|digas|estes))\b",
    
    # --- ESTRATEGIAS Y RAPPORT MANAGEMENT ---
    "Vocativo_negativo_personalizado": r"\b(put[oa]s?|asqueros[oa]s?|estupid[oa]s?|imbecil(es)?|analfabetos?|ridicul[oa]s?|nac[oa]s?|igualad[oa]s?|jodid[oa]s?|viej[oa]s? repugnante(s)?|chocorrol|perdedor(es)?|mediocres?)\b",
    "Asercion_negativa_personalizada": r"\b(eres verdaderamente un pendejo|eres un hipocrita|eres una zorra|no puedes hacer nada bien|pide perdon por ser tan cutre|de educacion y cortesia no tienes|ser whitexican no es solo un color|el kks siempre tan kk|son tan pueblerinos|das lastima|estas bien pendejo|eres (un|una) inutil)\b",
    "Referencias_negativas_personalizadas": r"\b(tu boca apestosa|me das asco|ni en roblox te sabes vestir( bien)?|ag[u]ite me das|se ve bien jodid[oa]|se ve bien jodida dona cara de carton|tu cara de pendejo)\b",
    "Referencias_negativas_personalizadas_en_tercera_persona": r"\b(es[ea] tont[oa]|ell[oa] esta tarad[oa]|no tienes ni tantita madre|viej[oa] que siempre fue una mierda|lo dice el anciano|malparido de mierda|es[ea] idiot[a]|pinche loc[oa])\b",
    "Criticas_o_Quejas": r"\b(no te metas en p[e3]dos ajenos|ya no respetas ni a tu jefa|haces y dices puras mamadas|que mujer tan cutre|la india maria dice|pinches llorones|siempre cagandola)\b",
    "Presuposiciones_o_preguntas_desagradables": r"\b(pobre pendejo|color mole|ya te creiste muy chingon|me crees pendeja|todo es tan cutre|gacho ser racista|gente se dara cuenta de quien es el estupido|te pagan por ser (tan)? pendejo)\b",
    "Condescendencia": r"\b(haces puras mamadas|parece sarcasmo|de tanto pensar|se les aparecio su padre|estas bien prieta|como si ser clasista fuera malo|pobre viejo rancio|la marca mas excluyente|perro oso|lo que callan los arios|estas sordo o que|la cagada va en la taza|no entiendes ni con manzanas|pobrecit[oa])\b",
    "Reforzadores": r"\b(me entiendes|te queda claro|escuchaste bien|a ver si asi entiendes|captas)\b",
    "Silenciadores": r"\b(callate( la puta boca)?|cierra el pico|nadie te pidio tu opinion|tu no entiendes|dices puras mamadas|callese( señora| viejo)?|cerrando el pico|cerrando el ocio|pareces perico|no me vengas con mamadas|no digas mamadas|shh|shutup|silencio)\b",
    "Despidos": r"\b(te me vas ahorita|g[u]ey abrete|largate( de aqui)?|muevete|a llorar a otro lado|ya te estas yendo|fuera( de aqui)?|no te quiero( ver)?|desaparecete|a chingar( a)? sus( pinches)? madres|vete a la (verga|cola)|mudate de continente|lloreria|abriend[oe]|llegale|a chillar a su casa|de puntitas a la v|fuchi|bai)\b",
    "Amenazas": r"\b(te (veo en la calle|doy en la madre|parto tu madre|va a cargar)|te voy a (madrear|romper|chingar|encontrar)|romper tu madre|conmigo no te metes|ya veras)\b",
    "Maldiciones_palabrotas": r"\b(jodet[ea]|mueret[ea]|que te parta un rayo|vete al (carajo|infierno|diablo)|vete a la (puta|mierda)|nunca hubieras nacido|ojala te mueras|pudret[ea]|maldit[oa] seas|chinga a tu madre)\b",
    "Imagen_De_Calidad": r"\b((eres|estas) (bien |un )?(pendejo|estupido|inutil|asco|nefast[oa]|gorda|chafas?)|pareces (una vaca|un mono)|no das una|chafas?|mal[oa])\b",
    "Imagen_De_Identidad_Social": r"\b(eres un[oa]? (indio|naco|godinez|fresa|hijo de puta|chairo|fifi)|te falta barrio|tu tez humilde)\b",
    "Imagen_De_Relacion": r"\b(eres (un |una )?(tonto|lamebotas|culero|zorra|vendido)|tu (papa|mama) es un[oa]?|profe es bien barco)\b",
    "Derechos_de_sociabilidad_Equidad": r"\b(que trasero|me la pelas|callate|perro tu no opinas|aqui mando yo|obedeces|quien te dio permiso de hablar|muerete indio|no vales nada|estas por debajo de)\b",
    "Derechos_de_sociabilidad_Asociacion": r"\b(es un progre|no eres bienvenido|fuera de aqui|largate|aqui no te queremos|no te juntes con nosotros|no eres de los nuestros|das un chingo de asco)\b",
    
    # --- DOMINANCIA E INTERACCIÓN ---
    "Hablante_dominante": r"\b(yo|mi|mando yo|conmigo|para mi)\b",
    "Oyente_dominante": r"\b(tu|ustedes|te|ti|tu mismo|contigo)\b",
    "Hablante_Oyente_dominante": r"\b(nosotros|nuestro|nosotras|nuestras)\b",
    "Tercero_dominante": r"\b(el|ella|ellos|ellas|su|sus)\b",
    "Oyente_Hablante_dominante": r"\b(nosotros|ustedes)\b",
    
    # --- RISAS ---
    "Risas": r"\b(jaja[ja]*|jeje[je]*|jiji[ji]*|haha[ha]*|jsjs[js]*|ja(ja)+|je(je)+|ji(ji)+|ha(ha)+|js(js)+|xd|lmao|lol)\b",
}

# ==============================================================================
# 2. DEFINICIÓN DE COLUMNAS MODIFICADORAS (Descriptivas y Estructurales)
# ==============================================================================
# Estas son las columnas que actúan como "propiedades" de la frase. 
columnas_modificadores = [
    "Cuantificadores_Adverbios_Cualidad",
    "Cuantificadores_Sustantivos_colectivos_o_determinantes_numerales_Entidad",
    "Determinantes_exclamativos",
    "Comparaciones_hiperbolicas",
    "Subjuntivo",
    "Imperativo",
    "Reforzadores",
    "Hablante_dominante",
    "Oyente_dominante",
    "Hablante_Oyente_dominante",
    "Tercero_dominante",
    "Oyente_Hablante_dominante",
    "Risas"
]

# El resto del código calcula los núcleos solitos:
# columnas_nucleo = [cat for cat in diccionario_lexemas.keys() if cat not in columnas_modificadores]

# 2. Las columnas núcleo son todas las demás (las que SÍ marcan una agresión directa/indirecta)
columnas_nucleo = [cat for cat in diccionario_lexemas.keys() if cat not in columnas_modificadores]

# ==============================================================================
# FASE 2: MOTOR DE CLASIFICACIÓN
# ==============================================================================
print(">>> FASE 2: Aplicando reglas léxicas (Regex) al corpus...")

def detectar_lexemas(texto_original):
    # Solo usamos el texto sustituido para detección; no se sobreescribe Texto_Comentario.
    t = sustituir_emojis_prioridad(texto_original)
    t = normalizar_texto(t)
    resultados = {}
    for categoria, patron_regex in diccionario_lexemas.items():
        resultados[categoria] = 1 if re.search(patron_regex, t) else 0
    return pd.Series(resultados)

# Generamos las 67 columnas de clasificación
todas_las_categorias = list(diccionario_lexemas.keys())
df[todas_las_categorias] = df['Texto_Comentario'].apply(detectar_lexemas)

# ==============================================================================
# FASE 3: ETIQUETA DE CONTROL Y LIMPIEZA DE FALSOS POSITIVOS
# ==============================================================================
print(">>> FASE 3: Evaluando núcleos vs. modificadores...")

# La categoría 62: Es "Insulto" SOLO si tiene al menos 1 categoría NÚCLEO detectada.
df['Es_Insulto'] = (df[columnas_nucleo].sum(axis=1) > 0).astype(int)

# Limpieza: Si Es_Insulto == 0, significa que cualquier modificador detectado es un falso positivo que no acompañaba a un insulto. Los reseteamos a 0.
for col in columnas_modificadores:
    df.loc[df['Es_Insulto'] == 0, col] = 0

df.to_csv(ARCHIVO_SALIDA, index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print(f"[*] PROCESO DICCIONARIO (67 CATS + 1) FINALIZADO CON ÉXITO.")
print(f"[*] Total de columnas de análisis: {len(todas_las_categorias) + 1}")
print(f"[*] Comentarios neutros ignorados (Es_Insulto = 0): {(df['Es_Insulto'] == 0).sum()}")
print(f"[*] Insultos confirmados (Es_Insulto = 1): {df['Es_Insulto'].sum()}")
print(f"[*] Resultados guardados en: {ARCHIVO_SALIDA}")
print("="*60)

## Francés

In [ ]:
import pandas as pd
import re
import unicodedata

# ==============================================================================
# FASE 0: PREPROCESAMIENTO, LIMPIEZA Y NORMALIZACIÓN
# ==============================================================================
print(">>> FASE 0: Limpiando el dataset de ruidos y nulos...")
ARCHIVO_ENTRADA = "dataset_comentarios_frances.csv"
ARCHIVO_SALIDA = "corpus_clasificado_DICCIONARIO_FRANCES.csv"

diccionario_emojis_prioridad = {
    # Risas y Burla
    "😂": " mdr ",
    "🤣": " ptdr ",
    "💀": " c'est mort ",
    "😭": " pleurnichard ",
    "🤡": " bouffon ",
    "🥱": " ennuyeux ",

    # Escatología y Asco
    "💩": " merde ",
    "🤮": " vomi ",
    "🤢": " degueulasse ",
    "🗑️": " poubelle ",
    "🗑": " poubelle ",

    # Animalización / Insultos
    "🐒": " singe ",
    "🦍": " gorille ",
    "🐷": " porc ",
    "🐖": " cochon ",
    "🐀": " rat ",
    "🐄": " vache ",
    "🐶": " chien ",
    "🐍": " serpent ",
    "🐑": " mouton ",

    # Agresión y Gestos
    "🖕": " va te faire foutre ",
    "🤬": " putain ",
    "😡": " colere ",
    "🤫": " ferme la ",

    # Intelecto
    "🧠": " cerveau ",
    "🤓": " intello ",
}

def contiene_texto_o_emoji_prioritario(texto):
    t = str(texto)
    tiene_alfanumerico = bool(re.search(r'[a-zA-Z0-9]', t))
    tiene_emoji_prioritario = any(emoji in t for emoji in diccionario_emojis_prioridad)
    return tiene_alfanumerico or tiene_emoji_prioritario

def normalizar_texto(texto):
    """
    Elimina acentos y convierte a minúsculas. 
    Mantiene símbolos como @ o * para detectar evasiones de censura.
    """
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def sustituir_emojis_prioridad(texto):
    t = str(texto)
    for emoji, reemplazo in diccionario_emojis_prioridad.items():
        t = t.replace(emoji, reemplazo)
    return t

try:
    df = pd.read_csv(ARCHIVO_ENTRADA)
    df = df.dropna(subset=['Texto_Comentario'])
    df = df[df['Texto_Comentario'].str.strip() != "<redacted_mention>"]
    df = df[df['Texto_Comentario'].apply(contiene_texto_o_emoji_prioritario)]
    df = df.drop_duplicates(subset=['Texto_Comentario'])
    print(f"[*] Limpieza completada. Registros útiles para procesar: {len(df)}")
except Exception as e:
    print(f"[X] Error en carga/limpieza: {e}")
    df = pd.DataFrame({"Texto_Comentario": ["T'es vraiment un gros bâtard", "Très belle université", "Ferme ta gueule fdp"]})

# ==============================================================================
# FASE 1: DICCIONARIO DE LEXEMAS (ARGOT FRANCÉS)
# ==============================================================================
print(">>> FASE 1: Construyendo reglas basadas en lexemas para las categorías en francés...")

diccionario_lexemas = {
    # --- MECANISMOS DE CONSTRUCCIÓN ---
    "Acto_explicito": r"\b(imbecile?s?|connards?|connasses?|abrutis?|p[\*u]tes?|merdes?|cons?|connes?|salopes?|batards?|pouffiasses?|tocards?|bouffons?|boloss(es)?)\b",
    "Enunciados_declarativos_apelativos": r"\b(t'es|tu es|vous etes|c'est)\s+(un|une|des)\s+(faineants?|bon a rien|malade(s)? mental|pitoyables?|merdes?|laches?|con|connard|idiot)\b",
    
    # Subcategorías de Rechazo
    "Actos_directivos_de_rechazo_con_formulas_imprecatorias": r"\b(va au diable|creve|meurt|puisses tu (crever|mourir)|sois (maudit|damne))\b",
    "Actos_directivos_de_rechazo_con_negaciones_metaforicas": r"\b(ne sois pas un|casse pas les couilles|arrete de (faire chier|casser les couilles)|ne fais pas le)\b",
    "Actos_directivos_de_rechazo_con_metaforas_directas": r"\b(va te faire (mettre|cuire un oeuf|enculer|foutre|cuire le cul|voir))\b",
    "Actos_directivos_de_rechazo_con_mandato_de_silencio": r"\b(ferme (la|ta gueule|ton clapet)|tais(-| )?toi|chut|boucle la|clouez leur le bec|rabaisse la caquet)\b",
    "Actos_directivos_de_rechazo_con_exhortaciones_de_alejamiento": r"\b(degage|barre(-| )?toi|fiche le camp|retourne dans|casse(-| )?toi|tire(-| )?toi|hors de ma vue|fous le camp)\b",
    
    "Abreviaturas": r"\b(fdp|ftg|ntm|pd|tdc|tgl|tdb|bz|vtff|stfu|mdr|ptdr|btdr|osef|balek|ndm)\b",
    "Cambio_de_modalidad": r"\b(t'as pas de|est(-| )?ce que tu|comment on peut|tu penses que c'est|cerveau|debile\?)\b",
    "Contenido_lexico": r"\b(faux(-| )?cul|negre de maison|commere|sangsue|tete de (n[oe]ud|con|bite|fion|mule)|face de (rat|pet|cul)|espece de)\b",
    "Deixis_discursiva": r"\b(la faire a l'envers|te la casser|tu fais pitie|tu saoules|c'est mort)\b",
    "Eufemismos": r"\b(pas la lumiere|limite|special|pas net|casse|fini a la pisse|berce trop pres du mur|pas invente l'eau|roue du feu|reflexion de bebe|lache l'affaire|saucisson|relou|en trop)\b",
    "Omision_de_la_groseria": r"\b(fils de \.\.\.|va te faire \.\.\.|espece de \.\.\.|ta m\.\.\.|put\.\.\.|je vais te d\.\.\.)\b",
    "Posesivos_enfatizadores": r"\b(ta (mere|gueule|race|daronne|teub|grand mere)|ton (cul|pere|froc|reuf)|tes (morts|couilles))\b",
    "Humor": r"\b(le genie de la|prix nobel de|darwin|t'es drole pour un|rire de toi|intellectuel|champion du monde|prix du ridicule)\b",
    "Ironia_polar_o_contrastiva": r"\b(quelle subtilite|t'es un genie|bravo l'artiste|super intelligent|quelle lumiere|vrai champion)\b",
    "Ironia_de_personificacion": r"\b(le monde est contre|mon metabolisme|mon algorithme a dit|pauvre chou|tu es la victime)\b",
    "Ironia_de_falsa_cortesia": r"\b(merci pour cette intervention|merci de ton avis|avec tout mon respect|je t'en prie le genie|tres pertinent)\b",
    
    # --- TAXONOMÍA (SEXOTIPOS Y ONTOTIPOS) ---
    "Sexualidad_Mujer": r"\b(putes?|salopes?|petasses?|pouffiasses?|chaudasses?|michtos?|tchoin|catin|nymphos?|cagoles?|marie couche toi la|garces?)\b",
    "Sexualidad_Hombre": r"\b(pedes?|tapettes?|tarlouzes?|pedales?|fiottes?|tantouzes?|travelos?|encules?)\b",
    "Sexualidad_Persona_no_binaria": r"\b(trans|travelos?|hermaphrodite|monstre|mutant|alien|bizarre)\b",
    "Anatomia_genital": r"\b(bites?|couilles?|chibres?|chattes?|culs?|trou du cul|zgeg|teub|zizi|minou|moule|queue|schnek|pine|chnek|burnes)\b",
    "Filiacion": r"\b(fils de (pute|chien|lache|garce)|batards?|nique ta mere|ta daronne|enfant de putain|enfant de (salope|catin)|enfoires?)\b",
    "Capacidad_intelectual": r"\b(cons?|connes?|debiles?|teubes?|idiots?|mongols?|attardes?|boloss(es)?|cassos?|gogols?|abrutis?|betes?|ignorants?|golmons?|trisos?|cretins?|imbeciles?)\b",
    "Caracter": r"\b(flemmards?|branleurs?|bon(s)? a rien|glandeurs?|laches?|chialeurs?|fragiles?|pleurnichards?|peureux|mous?|galeriens?|parasites?|faineants?|lourds?)\b",
    "Conducta": r"\b(connards?|connasses?|fouteurs? de merde|crevards?|ordures?|bouffons?|enflures?|raclures?|pourritures?|charognards?|chacals?|petasses?|casse couilles)\b",
    "Moralidad": r"\b(hypocrites?|faux culs?|balances?|poucaves?|traitres?|viperes?|rats?|pinces?|menteurs?|voleurs?|gratteurs?|snitchs?|vendus?|escrocs?|ordures?)\b",
    "Apariencia_fisica": r"\b(moches?|thons?|gros|grolards?|nabots?|laids?|degueulasses?|monstres?|nains?|obeses?|boudins?|planche a pain|binoclards?|bigleux|squelettes?|gueule de cul)\b",
    "Forma_de_vestir": r"\b(griffue|poilleux|pouilleux|ringards?|clodos?|fagote|sac a patates|bling[- ]bling|habille comme un clochard|criard|souillon|crasseux|sagouin)\b",
    "Edadismo": r"\b(boomers?|vieux schnocks?|fossiles?|croulants?|dinosaures?|chibanis?|ancetres?|darons?|vieillards?|debris?|morveux|gamins?|pisseuses?|jeunots?)\b",
    "Afectividad_empatia": r"\b(egoistes?|sans c[oe]ur|sans coeur|sans race|sociopathes?|insensibles?|sans pitie|froids?|ameres?|coeur de pierre)\b",
    "Zoomorfismo_animalizacion": r"\b(rats?|sangsues?|langue de vipere|chaud lapin|chiennes?|thon|girafe|carapace|singe( a lunettes)?|bourrin|crack panther|blaireau|andouille|poule mouillee|tete de mule|tocard|parasite|cochon|jument|grenouille|raton|cabot|clebs|chiens?|truie|serpent|couleuvre|souris|vermine|vautour|corbeau|pie|perroquet|poulet|cafards?|vache|genisse|ane|bourrique|mule|hyene|pauvre here|cerveau de moineau|sanglier)\b",
    "Escatologia": r"\b(merdes?|chiasses?|caca|etrons?|pisses?|vomi(s)?|dechets?|bouses?|pourritures?|fientes?|crottes?)\b",
    "Estatus_social": r"\b(prolos?|bourgeois|bourges?|fils de bourge|sdf|miserables?|fils de papa|paysans?|gueux|beaufs?|ploucs?|bouseux)\b",
    "Ideologia": r"\b(rustre|caillera|communiste|petit[- ]bourgeois|macroniste|fachos?|neonazis?|reac|extreme[- ]droite|anti[- ]france|francophobe|extreme[- ]gauche|nazis?|gauchistes?|gocho|fascistes?|droitards?|gaucher|gauchiasse|socialos|anarchiste|bolcho|woke|integriste|faf|bigot|calotin|cureton|grenouille de benitier|laicard|populiste|demagogue|conspirationniste|vendu|opportuniste|complotiste|bobo|feministe|collabos?|neoliberaliste)\b",
    "Profesion": r"\b(flics?|porcs?|keufs?|condes?|pions?|prof(s)? de merde|gratte papiers?|baqueux|poulets?|bureaucrates?|faineants? de fonctionnaires?)\b",
    "Racismo": r"\b(negre( de maison)?|race|niakoue|bougnoules?|negresse|noir|juif|youpin|guenon|moricaud|noiraud|macaque|bicot|crouille|arabe de service|beur|rebeu|nordaf|yid|feuj|ching chong|chinetoque|jaunasse|blanc[- ]bec|toubab( fouf)?|cefran)\b",
    "Xenofobia": r"\b(bledards?|gwer|babtou|arabe|chinois|chinoise|chintok|rital|bouche|frouze|chiabrena|tsniout|immigres?|sans[- ]papiers?|voleur de boulot|espingouin|macaroni|fritkot|portos|portoche|manouch|kahlouch|arbouch|angliche|boche|chleuh|tzigane|rom|amerloque|yankee|amerlo)\b",
    
    # --- MODIFICADORES E INTENSIFICADORES ---
    "Cuantificadores_Adverbios_Cualidad": r"\b(?:tres|trop|grave|super|mega|archi|ultra|hyper|vachement|tellement|bien|vraiment|excessivement|fort)\b",
    "Cuantificadores_Sustantivos_colectivos_o_determinantes_numerales_Entidad": r"\b(?:bande de|espece de|tas de|ramassis de|horde de|groupe de|tous les|pleins de|un paquet de|une (bande|chiee) de)\b",
    "Determinantes_exclamativos": r"(?:[¡!¿?]*\b(?:quel|quelle|quels|quelles|comme)\b)",
    "Comparaciones_hiperbolicas": r"\b(?:plus\s+\w+\s+que|pire\s+que|moins\s+\w+\s+que|aussi\s+\w+\s+que(?: de| la| le)?|comme un[e]? (?:vache|porc|chien|merde|con|balai|pou))\b",
    "Comparaciones_verlan": r"\b(reuf|teub|meuf|youpin|chomeurs|keuf|beur|rebeu|cefran|feuj|ouf)\b",
    "Subjuntivo": r"\b(?:j'espere que|pourvu que|que tu|puisses tu)\b",
    "Imperativo": r"\b(?:va te|mange|ferme|nique|creve|degage|barre|fiche|casse|tais|arrete)\b",
    
    # --- ESTRATEGIAS Y RAPPORT MANAGEMENT ---
    "Vocativo_negativo_personalizado": r"\b(?:sale|espece de|putain de|gros|pauvre|petit(e)?)\b",
    "Asercion_negativa_personalizada": r"\b(?:tu sers a rien|t'es vraiment|tu fais honte|tu es un(e)? vrai(e)?|c'est honteux (de|pour)|t'es (vraiment |juste )?(une merde|un con|idiot))\b",
    "Referencias_negativas_personalizadas": r"\b(?:ta tete|tu degoutes|ton haleine|ta gueule|ta face|ton visage|ton nez)\b",
    "Referencias_negativas_personalizadas_en_tercera_persona": r"\b(?:cette conne|ce mec|ce boloss|cette meuf( est)?|ce gars|il/elle est vraiment|le/la pauvre)\b",
    "Criticas_o_Quejas": r"\b(?:tu fais chier|tu me saoules|vous (faites|nous) chier|c'est (chiant|nul|n'importe quoi)|(honte|dommage) a la|honteux|decevant)\b",
    "Presuposiciones_o_preguntas_desagradables": r"\b(?:tu te prends pour qui|comment on peut|t'as un probleme|c'est quoi ton probleme|qui t'a demande|tu (es|crois) serieux)\b",
    "Condescendencia": r"\b(?:pauvre type|pauvre (fou|fille|gars)|mon petit|ma pauvre|tu fais pitie|tu m'amuses|tu me fais rire|il a cru que)\b",
    "Reforzadores": r"\b(?:tu comprends|c'est clair|t'as capte|tu vois|compris|tu m'entends|non mais allo)\b",
    "Silenciadores": r"\b(?:ta gueule|ferme la|tais toi|chut|motus|silence|on t'a pas sonne|tu te tais)\b",
    "Despidos": r"\b(?:degage|casse toi|barre toi|tire toi|vas t'en|fous le camp|cassez vous|ciao|au revoir|dehors)\b",
    "Amenazas": r"\b(?:je vais te (casser|frapper|tuer|niquer|trouver|retrouver)|tu vas voir|fais attention a toi|gare a toi|ca va mal finir)\b",
    "Maldiciones_palabrotas": r"\b(?:va au diable|creve|putain|merde|bordel|fait chier|fais chier|nique sa mere)\b",
    "Imagen_De_Calidad": r"\b(?:t'es (nul|une merde|incapable|bon a rien|incompetent|idiot|rate|minable)|sans talent)\b",
    "Imagen_De_Identidad_Social": r"\b(?:sale (bourge|prolo|beauf|racaille|paysan|gueux|banlieusard|intello))\b",
    "Imagen_De_Relacion": r"\b(?:(faux frere|vendu|traitre|suceur|leche cul|fayot|menteur|lache))\b",
    "Derechos_de_sociabilidad_Equidad": r"\b(?:ferme ta gueule|suce|agenouille toi|tu sers a rien|baisse les yeux|ici c'est moi qui|a genoux)\b",
    "Derechos_de_sociabilidad_Asociacion": r"\b(?:casse toi|hors de ma vue|retourne chez toi|t'es pas de (chez nous|notre niveau)|on veut pas de toi|fous le camp)\b",
    
    # --- DOMINANCIA E INTERACCIÓN ---
    "Hablante_dominante": r"\b(?:je|moi|mon|ma|mes|avec moi)\b",
    "Oyente_dominante": r"\b(?:tu|toi|ton|ta|tes|avec toi|vous|votre|vos)\b",
    "Hablante_Oyente_dominante": r"\b(?:nous|notre|nos)\b",
    "Tercero_dominante": r"\b(?:il|elle|ils|elles|son|sa|ses|leur|leurs)\b",
    "Oyente_Hablante_dominante": r"\b(?:nous|vous)\b",
    
    # --- RISAS ---
    "Risas": r"\b(?:mdr|ptdr|lol|xptdr|haha(ha)*|hehe(he)*|hihi(hi)*|mouhaha)\b"
}

# ==============================================================================
# 2. DEFINICIÓN DE COLUMNAS MODIFICADORAS (Descriptivas y Estructurales)
# ==============================================================================
# Estas son las columnas que actúan como "propiedades" de la frase. 
columnas_modificadores = [
    "Cuantificadores_Adverbios_Cualidad",
    "Cuantificadores_Sustantivos_colectivos_o_determinantes_numerales_Entidad",
    "Determinantes_exclamativos",
    "Comparaciones_hiperbolicas",
    "Comparaciones_verlan",
    "Subjuntivo",
    "Imperativo",
    "Reforzadores",
    "Hablante_dominante",
    "Oyente_dominante",
    "Hablante_Oyente_dominante",
    "Tercero_dominante",
    "Oyente_Hablante_dominante",
    "Risas"
]

# 2. Las columnas núcleo son todas las demás (las que SÍ marcan una agresión directa/indirecta)
columnas_nucleo = [cat for cat in diccionario_lexemas.keys() if cat not in columnas_modificadores]

# ==============================================================================
# FASE 2: MOTOR DE CLASIFICACIÓN
# ==============================================================================
print(">>> FASE 2: Aplicando reglas léxicas (Regex) al corpus en francés...")

def detectar_lexemas(texto_original):
    # Solo usamos el texto sustituido para detección; no se sobreescribe Texto_Comentario.
    t = sustituir_emojis_prioridad(texto_original)
    t = normalizar_texto(t)
    resultados = {}
    for categoria, patron_regex in diccionario_lexemas.items():
        resultados[categoria] = 1 if re.search(patron_regex, t) else 0
    return pd.Series(resultados)

# Generamos las columnas de clasificación (ahora 68 contando Comparaciones_verlan)
todas_las_categorias = list(diccionario_lexemas.keys())
df[todas_las_categorias] = df['Texto_Comentario'].apply(detectar_lexemas)

# ==============================================================================
# FASE 3: ETIQUETA DE CONTROL Y LIMPIEZA DE FALSOS POSITIVOS
# ==============================================================================
print(">>> FASE 3: Evaluando núcleos vs. modificadores...")

# La etiqueta Es_Insulto SOLO se activa si tiene al menos 1 categoría NÚCLEO detectada.
df['Es_Insulto'] = (df[columnas_nucleo].sum(axis=1) > 0).astype(int)

# Limpieza: Si Es_Insulto == 0, significa que cualquier modificador detectado es un falso positivo. Los reseteamos a 0.
for col in columnas_modificadores:
    df.loc[df['Es_Insulto'] == 0, col] = 0

df.to_csv(ARCHIVO_SALIDA, index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print(f"[*] PROCESO DICCIONARIO FRANCÉS FINALIZADO CON ÉXITO.")
print(f"[*] Total de columnas de análisis: {len(todas_las_categorias) + 1}")
print(f"[*] Comentarios neutros ignorados (Es_Insulto = 0): {(df['Es_Insulto'] == 0).sum()}")
print(f"[*] Insultos confirmados (Es_Insulto = 1): {df['Es_Insulto'].sum()}")
print(f"[*] Resultados guardados en: {ARCHIVO_SALIDA}")
print("="*60)

# Muestra de comentarios

In [ ]:
import pandas as pd

PUESTOS = """
Si crees que haces algo mediante esta pagina tas bien wey, vete a hablar con los estudiantes
Cómo se llama la gorda
Oye Samantha Gallegos Flores como sientes la funa despúes de andar de ofrecida y andar con tu jefe en tv buap, ya se te bajó la calentura o todavía no te queda claro con quienes te metiste 😬
Pues muy pendejos qué dejaran qué sé infiltraron en el movimiento
Ningún estudiante la quiere como rectora, RIDICULOS.
Muestre la cara, un diálogo frente a frente es lo que se está pidiendo. No que nos traten como delincuentes.
Nmms y si tenemos que ir a recoger documentos que solo te dan en físico??? Valen para pura madre, al fin que ni quería entrar a la maestría el siguiente semestre, neta vayanse ALV con su 💩 paro 👨🏻‍🎓👨🏻‍🎓👨🏻‍🎓👨🏻‍🎓
"""

# Asegúrate de poner el nombre correcto de tu archivo clasificado
ARCHIVO_ENTRADA = "corpus_clasificado_DICCIONARIO_61_CATS.csv"

def normalizar_comentario(texto):
    # Normaliza espacios y mayúsculas/minúsculas para comparar de forma robusta.
    return " ".join(str(texto).split()).strip().casefold()

try:
    # 1. Cargar el dataset ya clasificado
    df = pd.read_csv(ARCHIVO_ENTRADA)
    
    # 2. Filtrar SOLO los comentarios que el sistema detectó como INSULTOS
    if 'Es_Insulto' in df.columns:
        df_insultos = df[df['Es_Insulto'] == 1].copy()
    else:
        print(f"[X] Error: La columna 'Es_Insulto' no se encontró en el DataFrame. Asegúrate de que el proceso de clasificación se haya completado correctamente.")
        df_insultos = pd.DataFrame()

    # 3. Excluir comentarios listados en PUESTOS (cada salto de línea = 1 comentario)
    comentarios_excluir = {
        normalizar_comentario(linea)
        for linea in PUESTOS.splitlines()
        if linea.strip()
    }

    if not df_insultos.empty and 'Texto_Comentario' in df_insultos.columns:
        total_antes = len(df_insultos)
        df_insultos = df_insultos[
            ~df_insultos['Texto_Comentario'].astype(str).map(normalizar_comentario).isin(comentarios_excluir)
        ]
        print(f"[*] Comentarios excluidos por estar en PUESTOS: {total_antes - len(df_insultos)}")

    print(f"[*] Total de insultos detectados en el corpus (tras exclusión): {len(df_insultos)}")
    
    # 4. Extraer muestra aleatoria de N comentarios
    # Usamos random_state=42 para que la muestra sea reproducible.
    TAMAÑO_MUESTRA = 200
    if len(df_insultos) > TAMAÑO_MUESTRA:
        muestra = df_insultos.sample(n=TAMAÑO_MUESTRA, random_state=42)
    else:
        muestra = df_insultos
        print(f"[!] Hay menos de {TAMAÑO_MUESTRA} insultos, mostrando todos.")

    # 5. Definir el orden exacto de las columnas que se imprimirán.
    columnas_ordenadas = [
        "Acto_explicito", "Enunciados_declarativos_apelativos",
        "Actos_directivos_de_rechazo_con_formulas_imprecatorias",
        "Actos_directivos_de_rechazo_con_negaciones_metaforicas",
        "Actos_directivos_de_rechazo_con_metaforas_directas",
        "Actos_directivos_de_rechazo_con_mandato_de_silencio",
        "Actos_directivos_de_rechazo_con_exhortaciones_de_alejamiento",
        "Abreviaturas", "Cambio_de_modalidad", "Contenido_lexico",
        "Deixis_discursiva", "Eufemismos", "Omision_de_la_groseria",
        "Posesivos_enfatizadores", "Humor", "Ironia_polar_o_contrastiva",
        "Ironia_de_personificacion", "Ironia_de_falsa_cortesia",
        "Sexualidad_Mujer", "Sexualidad_Hombre", "Sexualidad_Persona_no_binaria",
        "Anatomia_genital", "Filiacion", "Capacidad_intelectual", "Caracter",
        "Conducta", "Moralidad", "Apariencia_fisica", "Forma_de_vestir",
        "Edadismo", "Afectividad_empatia", "Zoomorfismo_animalizacion",
        "Escatologia", "Estatus_social", "Ideologia", "Profesion", "Racismo",
        "Xenofobia", "Cuantificadores_Adverbios_Cualidad",
        "Cuantificadores_Sustantivos_colectivos_o_determinantes_numerales_Entidad",
        "Determinantes_exclamativos", "Comparaciones_hiperbolicas", "Subjuntivo",
        "Imperativo", "Vocativo_negativo_personalizado",
        "Asercion_negativa_personalizada", "Referencias_negativas_personalizadas",
        "Referencias_negativas_personalizadas_en_tercera_persona",
        "Criticas_o_Quejas", "Presuposiciones_o_preguntas_desagradables",
        "Condescendencia", "Reforzadores", "Silenciadores", "Despidos",
        "Amenazas", "Maldiciones_palabrotas", "Imagen_De_Calidad",
        "Imagen_De_Identidad_Social", "Imagen_De_Relacion",
        "Derechos_de_sociabilidad_Equidad", "Derechos_de_sociabilidad_Asociacion",
        "Hablante_dominante", "Oyente_dominante", "Hablante_Oyente_dominante",
        "Tercero_dominante", "Oyente_Hablante_dominante", "Risas",
        "Es_Insulto"
    ]

    print("\n" + "="*80)
    print(" Comentarios ".center(80, "="))
    print("="*80 + "\n")
    
    # 6. Iterar y mostrar solo las etiquetas activadas (valor = 1)
    for i, (index, row) in enumerate(muestra.iterrows(), 1):
        print(f"[{i}/{len(muestra)}] ID: {row['ID_Comentario']} | PÁGINA: {row.get('Pagina_Origen', 'Desconocida')}")
        print(f"TEXTO: {row['Texto_Comentario']}")

        etiquetas_activas = []
        for col in columnas_ordenadas:
            if col in row:
                if int(row[col]) == 1:
                    etiquetas_activas.append(col)
            else:
                print(f"[!] Advertencia: La columna '{col}' no se encontró en el DataFrame.")

        if etiquetas_activas:
            print(f"ETIQUETAS ACTIVAS: {', '.join(etiquetas_activas)}")
        else:
            print("ETIQUETAS ACTIVAS: Ninguna")

        print("-" * 80)

except FileNotFoundError:
    print(f"[X] No se encontró el archivo '{ARCHIVO_ENTRADA}'. Asegúrate de haber corrido la clasificación primero.")
except Exception as e:
    print(f"[X] Ocurrió un error: {e}")